# deckard Layers Walkthrough

This notebook explains the scripts in `deckard/layers`, with emphasis on `optimize.py` and the Hydra override path it depends on.

It is meant to be read alongside the more focused notebooks:

- [hydra.ipynb](hydra.ipynb) for composition and override syntax
- [optuna.ipynb](optuna.ipynb) for studies, trials, samplers, pruners, and storage
- [artifacts.ipynb](artifacts.ipynb) for end-to-end output and artifact handling
- [lifelines.ipynb](lifelines.ipynb) for survival-mode workflows
- [anjana.ipynb](anjana.ipynb) for a broader API tour across data, model, attack, and plotting layers

The goal here is narrower: show how the layer entrypoints fit together, how Hydra configuration flows into them, and where the optimize callback writes runtime files.

In [1]:
import inspect
from pathlib import Path

from deckard.layers import optimize, plot, progress_bar

layer_root = Path("../../deckard/layers").resolve()
layer_files = [
    layer_root / "optimize.py",
    layer_root / "plot.py",
    layer_root / "progress_bar.py",
]

print("Layer scripts:")
for path in layer_files:
    print("-", path.name)

print("\nOptimize callback:", optimize.OptunaStudyCallback.__name__)
print("Plot entrypoint module:", plot.__name__)
print("Progress bar helper module:", progress_bar.__name__)

print("\nPublic optimize helpers:")
for name in [
    "_resolve_multirun_paths",
    "_resolve_run_paths",
    "_normalize_mode_cfg",
    "_seed_experiment_uuid_for_current_trial",
]:
    print("-", name)

print("\nSource excerpt for OptunaStudyCallback:")
print(
    "\n".join(
        inspect.getsource(optimize.OptunaStudyCallback.on_compose_config).splitlines()
    )
)

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Layer scripts:
- optimize.py
- plot.py
- progress_bar.py

Optimize callback: DefaultOptimizerCallback
Plot entrypoint module: deckard.layers.plot
Progress bar helper module: deckard.layers.progress_bar

Public optimize helpers:
- _resolve_multirun_paths
- _resolve_run_paths
- _normalize_mode_cfg
- _seed_experiment_uuid_for_current_trial

Source excerpt for OptunaStudyCallback:
    def on_compose_config(self, config: DictConfig, **kwargs: Any) -> None:
        """
        Prepare per-job naming, output paths, and persist params.yaml.

        Args:
            config: Hydra config for the current job.
            **kwargs: Additional keyword arguments.

        Note:
            Writes params.yaml after resolving file paths.
        """
        self._configure_policy(config)
        hydra_cfg = HydraConfig.get()
        _normalize_mode_cfg(config, hydra_cfg, include_file_paths=True)
        _seed_experiment_uuid_for_current_trial(
            hydra_cfg=hydra_cfg,
            experiment_

## How `deckard.layers.optimize` Works

`deckard.layers.optimize.OptunaStudyCallback` is the main bridge between Hydra and Optuna in Deckard.

At a high level, it:

- reads the composed Hydra config
- resolves per-run output paths such as `scores.json`, `params.yaml`, and log files
- creates or resumes the Optuna study from Hydra sweeper settings
- writes the final score payload back into the run directory

The most important idea is that the callback does not invent its own configuration shape. It consumes the same Hydra blocks used elsewhere in the notebook stack, so the behavior stays consistent with [hydra.ipynb](hydra.ipynb) and [optuna.ipynb](optuna.ipynb).

In [2]:
print("Hydra override targets used by optimize.py:")
for name in [
    "hydra.sweeper.study_name",
    "hydra.sweeper.storage",
    "hydra.sweeper.params",
    "hydra.run.dir",
    "hydra.sweep.dir",
    "hydra.sweep.subdir",
    "hydra.callbacks.deckard_optuna.score_file",
    "hydra.callbacks.deckard_optuna.params_file",
]:
    print("-", name)

print("\nOutput path resolution examples:")
print("- _resolve_multirun_paths(hydra_cfg) chooses run or sweep directories")
print("- _get_sweeper_cfg(hydra_cfg) normalizes DictConfig or plain dict access")
print(
    "- the callback then writes params.yaml and scores.json into the resolved directory"
)

Hydra override targets used by optimize.py:
- hydra.sweeper.study_name
- hydra.sweeper.storage
- hydra.sweeper.params
- hydra.run.dir
- hydra.sweep.dir
- hydra.sweep.subdir
- hydra.callbacks.deckard_optuna.score_file
- hydra.callbacks.deckard_optuna.params_file

Output path resolution examples:
- _resolve_multirun_paths(hydra_cfg) chooses run or sweep directories
- _get_sweeper_cfg(hydra_cfg) normalizes DictConfig or plain dict access
- the callback then writes params.yaml and scores.json into the resolved directory


## Cross-Links and Nearby Surfaces

The layer scripts are easiest to understand as a small workflow graph:

- `optimize.py` handles Hydra callbacks, Optuna studies, and per-run file resolution.
- `plot.py` turns config-composed experiments into plots and can infer experiment settings from Hydra configuration.
- `progress_bar.py` reads Hydra and DVC metadata to estimate how many studies or trials are expected.

For worked examples, jump to:

- [hydra.ipynb](hydra.ipynb) for composition and override syntax
- [optuna.ipynb](optuna.ipynb) for study and storage behavior
- [artifacts.ipynb](artifacts.ipynb) for end-to-end model artifacts
- [lifelines.ipynb](lifelines.ipynb) for survival-mode configuration
- [anjana.ipynb](anjana.ipynb) for a broader layered API demonstration

## How `deckard.layers.plot` Works

`deckard.layers.plot.plot_main` is a dispatcher for two plotting workflows that share the same `plot_params_file` input.

It first resolves the Hydra config into a normalized plot block, then chooses a backend:

- `yellowbrick` when an experiment config is present, because the plot is built from fitted models
- `seaborn` when no experiment config is present and the input is tabular or aggregated results in a data file

That split matters because the inputs and outputs are different. Yellowbrick works from a live experiment configuration and writes plots directly from model objects. Seaborn works from a results table and can render either a single plot or a list of plots from a YAML spec. The same `plot_params_file` key is used in both cases; the presence of an experiment decides which backend interprets it.

The helper functions in this module exist mostly to make that dispatch reliable:

- `_resolve_plot_args_from_cfg` merges top-level Hydra keys with the nested `plot` block
- `_extract_backend` decides whether the request should be treated as Yellowbrick or Seaborn
- `_load_experiment_config` and `_load_yaml` bring file-based config into memory
- `_normalize_yellowbrick_plots` keeps the `plots` argument consistent across scalar and list forms

See [artifacts.ipynb](artifacts.ipynb) and [anjana.ipynb](anjana.ipynb) for examples of this layer in a broader workflow.

In [3]:
import inspect

from deckard.layers import plot as plot_layer

print("plot.py helpers:")
for name in [
    "_resolve_plot_args_from_cfg",
    "_extract_backend",
    "_load_experiment_config",
    "_normalize_yellowbrick_plots",
    "plot_main",
]:
    print("-", name)

print("\nBackend decisions used by plot_main:")
for line in [
    "backend == 'auto' -> infer Yellowbrick when experiment config is present, otherwise Seaborn",
    "backend == 'yellowbrick' -> instantiate YellowbrickConfig or YellowbrickConfigList",
    "backend == 'seaborn' -> instantiate SeabornPlotConfig or SeabornPlotConfigList",
]:
    print("-", line)

print("\nShared plot_params_file behavior:")
print(
    "- Yellowbrick uses plot_params_file for plot_params when an experiment config exists"
)
print(
    "- Seaborn uses plot_params_file for single plot values or a list of plot specs when no experiment config exists"
)

print("\nSource excerpt for plot_main:")
print("\n".join(inspect.getsource(plot_layer.plot_main).splitlines()[:45]))

plot.py helpers:
- _resolve_plot_args_from_cfg
- _extract_backend
- _load_experiment_config
- _normalize_yellowbrick_plots
- plot_main

Backend decisions used by plot_main:
- backend == 'auto' -> infer Yellowbrick when experiment config is present, otherwise Seaborn
- backend == 'yellowbrick' -> instantiate YellowbrickConfig or YellowbrickConfigList
- backend == 'seaborn' -> instantiate SeabornPlotConfig or SeabornPlotConfigList

Shared plot_params_file behavior:
- Yellowbrick uses plot_params_file for plot_params when an experiment config exists
- Seaborn uses plot_params_file for single plot values or a list of plot specs when no experiment config exists

Source excerpt for plot_main:
def plot_main(cfg: Any) -> dict:
    """Execute plotting from either experiment config (Yellowbrick) or tabular results (Seaborn).

    Args:
    cfg: Plot config carrying experiment paths, data paths, backend choice,
        and backend-specific plotting parameters.
    title:
            Optional cust

## How `deckard.layers.survival` Works

`deckard.layers.survival.survival_main` is the entrypoint for survival analysis workflows.

The structure is simpler than `plot.py`, but it still has two distinct modes:

- plot-only mode when the config contains explicit plot specifications
- experiment mode when the config describes a full survival experiment

The workflow starts by pulling a `survival` block out of the Hydra config, validating the raw `data` and `model` specs, and then normalizing the model name into a canonical survival fitter label. That normalization is what makes user-facing aliases like `coxphfitter` or `weibullaftfitter` collapse into the small set of supported modes.

If plot-only mode is active, the module instantiates the experiment config, loads the dataset into a dataframe, and hands the result to the lifelines plotting plugin. If not, it instantiates the experiment config and runs the full workflow.

For a worked example, see [lifelines.ipynb](lifelines.ipynb).

In [4]:
from deckard.layers import survival as survival_layer

print("survival.py helpers:")
for name in [
    "survival_main",
    "_validate_raw_data_model_specs",
    "_coerce_survival_model_spec",
    "_has_plot_specification",
    "_run_plot_mode",
    "_run_experiment_mode",
]:
    print("-", name)

print("\nCanonical survival aliases handled by the layer:")
for name in [
    "cox",
    "weibull",
    "log-logistic",
    "log-normal",
    "aalen",
    "gamma",
    "exponential",
]:
    print("-", name)

print("\nSource excerpt for survival_main:")
print("\n".join(inspect.getsource(survival_layer.survival_main).splitlines()[:45]))

survival.py helpers:
- survival_main
- _validate_raw_data_model_specs
- _coerce_survival_model_spec
- _has_plot_specification
- _run_plot_mode
- _run_experiment_mode

Canonical survival aliases handled by the layer:
- cox
- weibull
- log-logistic
- log-normal
- aalen
- gamma
- exponential

Source excerpt for survival_main:
def survival_main(cfg: dict = None) -> dict:
    """Run survival workflow from Hydra-parsed config.

    Routes to either plot-only rendering or full experiment based on config.
    All values come from Hydra instantiation; there are no ad-hoc runtime args.

    Args:
        cfg: DictConfig or dict containing survival experiment configuration.
             Should have a 'survival' section where values resolve via
             instantiation to:
             - data: required DataConfig
             - model: required survival fitter name (e.g. weibull, cox)
             - target: required event column name
             - duration_col: required duration column name
    